In [4]:
import os

import numpy as np
import pandas as pd
import xarray as xr
from bluemath_tk.core.io import load_model
from bluemath_tk.core.operations import get_degrees_from_uv, get_uv_components
from bluemath_tk.datamining.mda import MDA
from bluemath_tk.datamining.pca import PCA
from bluemath_tk.interpolation.gps import ExactGPInterpolation
from utils.config import OFFSHORE_WAVES

whacs_config = OFFSHORE_WAVES["WHACS"]

In [5]:
num_mda_points = 500

In [6]:
mda_models = []
pca_models = []
gp_models = []

for goal in range(1, 11):
    print(f"Processing goal {goal}")
    metamodel_cases_goal = xr.open_dataset(
        f"SnapWave_CASES_METAMODEL/10mDepthAndBuoysPoints_{goal}.nc"
    )
    wavedir_u, wavedir_v = get_uv_components(metamodel_cases_goal.point_wavdir)
    metamodel_cases_goal["point_wavdir_u"] = wavedir_u
    metamodel_cases_goal["point_wavdir_v"] = wavedir_v
    metamodel_cases_goal["stations"] = range(177)
    metamodel_cases_df_for_goal = metamodel_cases_goal[
        ["tp", "wd", "ds", "zs"]
    ].to_dataframe()

    if os.path.exists(f"outputs/mda_{num_mda_points}/mda_model_{goal}.pkl"):
        mda = load_model(f"outputs/mda_{num_mda_points}/mda_model_{goal}.pkl")
    else:
        os.makedirs(f"outputs/mda_{num_mda_points}", exist_ok=True)
        mda = MDA(num_centers=num_mda_points)
        mda.fit(metamodel_cases_df_for_goal, directional_variables=["wd"])
        # mda.plot_selected_centroids();
        mda.save_model(f"outputs/mda_{num_mda_points}/mda_model_{goal}.pkl")
    mda_models.append(mda)

    if os.path.exists(f"outputs/pca_{num_mda_points}/pca_model_{goal}.pkl"):
        pca = load_model(f"outputs/pca_{num_mda_points}/pca_model_{goal}.pkl")
    else:
        os.makedirs(f"outputs/pca_{num_mda_points}", exist_ok=True)
        pca = PCA(n_components=0.95)
        pcs_ds = pca.fit_transform(
            data=metamodel_cases_goal.isel(case_num=mda.centroid_real_indices),
            vars_to_stack=["point_hm0", "point_tp", "point_wavdir_u", "point_wavdir_v"],
            coords_to_stack=["stations"],
            pca_dim_for_rows="case_num",
        )
        pca.save_model(f"outputs/pca_{num_mda_points}/pca_model_{goal}.pkl")
    pca_models.append(pca)

    if os.path.exists(f"outputs/gp_{num_mda_points}/gp_model_{goal}.pkl"):
        gp = load_model(f"outputs/gp_{num_mda_points}/gp_model_{goal}.pkl")
    else:
        os.makedirs(f"outputs/gp_{num_mda_points}", exist_ok=True)
        gp = ExactGPInterpolation()
        gp.fit(
            subset_data=mda.centroids[["tp", "wd", "ds", "zs"]],
            subset_directional_variables=["wd"],
            target_data=pca.pcs_df,
        )
        gp.save_model(f"outputs/gp_{num_mda_points}/gp_model_{goal}.pkl")
    gp_models.append(gp)

Processing goal 1
Processing goal 2
Processing goal 3
Processing goal 4
Processing goal 5
Processing goal 6
Processing goal 7
Processing goal 8
Processing goal 9
Processing goal 10


In [7]:
len(mda_models), len(pca_models), len(gp_models)

(10, 10, 10)

In [8]:
whacs_ds = xr.open_zarr(whacs_config["local_path"])
whacs_ds

<xarray.Dataset> Size: 1GB
Dimensions:                      (time: 394464, seapoint: 10)
Coordinates:
  * time                         (time) datetime64[ns] 3MB 1979-01-01 ... 202...
  * seapoint                     (seapoint) int32 40B 2323617 ... 2326141
Data variables: (12/41)
    cge                          (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    dir                          (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    dp                           (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    fp                           (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    hs                           (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    latitude                     (time, seapoint) float32 16MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    ...                           ...
    ptp3                         (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    spr                          (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    t01                          (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    t02                          (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    uwnd                         (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    vwnd                         (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
Attributes: (12/59)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_start:             1979-01-01T00:00:00
    time_coverage_end:               1979-01-31T23:00:00
    time_coverage_duration:          P30DT23H0M0S
    time_coverage_resolution:        P0DT1H0M0S

In [9]:
pd.read_csv(whacs_config["coordinates"])

,seapoint,longitude,latitude,goal
0,2326142,7.0,54.0,1
1,2326133,6.5,54.0,2
2,2326127,6.0,54.0,3
3,2326121,5.5,54.0,4
4,2326117,5.0,54.0,5
5,2325243,4.5,53.5,6
6,2324697,4.0,53.0,7
7,2324234,3.5,52.5,8
8,2334975,3.0,52.0,9
9,2323617,2.5,51.5,10


In [ ]:
whacs_ = xr.open_dataset("inputs/ERA5_forcing_goals_2000_2025.nc")
era5_forcing_partitions

<xarray.Dataset> Size: 184MB
Dimensions:    (time: 227928, goal: 10)
Coordinates:
  * time       (time) datetime64[ns] 2MB 2000-01-01 ... 2025-12-31T23:00:00
  * goal       (goal) int64 80B 1 2 3 4 5 6 7 8 9 10
    latitude   (goal) float64 80B ...
    longitude  (goal) float64 80B ...
Data variables: (12/20)
    wmb        (time, goal) float32 9MB ...
    swh        (time, goal) float32 9MB ...
    mwp        (time, goal) float32 9MB ...
    pp1d       (time, goal) float32 9MB ...
    mwd        (time, goal) float32 9MB ...
    wdw        (time, goal) float32 9MB ...
    ...         ...
    swh2       (time, goal) float32 9MB ...
    mwp2       (time, goal) float32 9MB ...
    mwd2       (time, goal) float32 9MB ...
    swh3       (time, goal) float32 9MB ...
    mwp3       (time, goal) float32 9MB ...
    mwd3       (time, goal) float32 9MB ...
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_edition:            1
    GRIB_subCentre:          0
    history:                 2025-05-05T16:51 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts

In [6]:
codec_dict_station_goal = {
    1: 23778,
    2: 23711,
    3: 23651,
    4: 23599,
    5: 23541,
    6: 23462,
    7: 23369,
    8: 23269,
    9: 23190,
    10: 23125,
}
codec_dict_station_goal

{1: 23778,
 2: 23711,
 3: 23651,
 4: 23599,
 5: 23541,
 6: 23462,
 7: 23369,
 8: 23269,
 9: 23190,
 10: 23125}

In [7]:
# SEA: Tp/Tm = 1.291 * gamma ^ -0.06241
# SWELL: Tp/Tm = 1.411 * gamma ^ -0.07972
sea_relation = 1.291 * 2**-0.06241
swell_relation = 1.411 * 6**-0.07972
print(sea_relation, swell_relation)

1.236342960063517 1.2231868290827768


In [8]:
reconstructed_months = []

# [
#     str(month)[:7]
#     for month in pd.date_range(start="2011-12-01", end="2011-12-31", freq="MS")
# ]
for month, month_str in zip(
    ["2011-12"],
    ["201112_500_tp"],
):
    era5_forcing_partitions_month = era5_forcing_partitions.sel(time=month)

    reconstructed_partitions = []
    reconstruction_for_each_goal = []

    for partition, variables in {
        "bulk": ["swh", "pp1d", "mwd", "wdw"],
        "sea": ["shww", "mpww", "mdww", "dwww"],
        "swell1": ["swh1", "mwp1", "mwd1", "dwps"],
        "swell2": ["swh2", "mwp2", "mwd2", "dwps"],
        "swell3": ["swh3", "mwp3", "mwd3", "dwps"],
    }.items():
        # reconstruction_for_each_goal = []
        for goal in range(1, 11):
            # First, set the proper variables
            if partition == "bulk":
                era5_ds_for_goal = era5_forcing_partitions_month.sel(goal=goal)[
                    variables
                ].rename(
                    {
                        "swh": "hs",
                        "pp1d": "tp",
                        "mwd": "wd",
                        "wdw": "ds",
                    }
                )
                era5_df_for_goal = era5_ds_for_goal.to_dataframe()
            elif partition == "sea":
                era5_ds_for_goal = era5_forcing_partitions_month.sel(goal=goal)[
                    variables
                ].rename(
                    {
                        "shww": "hs",
                        "mpww": "tp",
                        "mdww": "wd",
                        "dwww": "ds",
                    }
                )
                era5_df_for_goal = era5_ds_for_goal.to_dataframe()
                era5_df_for_goal["tp"] = era5_df_for_goal["tp"] * sea_relation
            elif partition == "swell1":
                era5_ds_for_goal = era5_forcing_partitions_month.sel(goal=goal)[
                    variables
                ].rename(
                    {
                        "swh1": "hs",
                        "mwp1": "tp",
                        "mwd1": "wd",
                        "dwps": "ds",
                    }
                )
                era5_df_for_goal = era5_ds_for_goal.to_dataframe()
                era5_df_for_goal["ds"] = 5
                era5_df_for_goal["tp"] = era5_df_for_goal["tp"] * swell_relation
            elif partition == "swell2":
                era5_ds_for_goal = era5_forcing_partitions_month.sel(goal=goal)[
                    variables
                ].rename(
                    {
                        "swh2": "hs",
                        "mwp2": "tp",
                        "mwd2": "wd",
                        "dwps": "ds",
                    }
                )
                era5_df_for_goal = era5_ds_for_goal.to_dataframe()
                era5_df_for_goal["ds"] = 5
                era5_df_for_goal["tp"] = era5_df_for_goal["tp"] * swell_relation
            elif partition == "swell3":
                era5_ds_for_goal = era5_forcing_partitions_month.sel(goal=goal)[
                    variables
                ].rename(
                    {
                        "swh3": "hs",
                        "mwp3": "tp",
                        "mwd3": "wd",
                        "dwps": "ds",
                    }
                )
                era5_df_for_goal = era5_ds_for_goal.to_dataframe()
                era5_df_for_goal["ds"] = 5
                era5_df_for_goal["tp"] = era5_df_for_goal["tp"] * swell_relation

            # Add water level
            era5_df_for_goal["zs"] = (
                xr.open_dataset(
                    f"inputs/CODEC/CODEC_{codec_dict_station_goal[goal]}.nc"
                )
                .sel(time=month)
                .waterlevel.values
            )
            era5_df_for_goal.dropna(inplace=True)

            # Predict in buoys
            prediction_ds_in_points = pca_models[goal - 1].inverse_transform(
                xr.Dataset(
                    {
                        "PCs": (
                            ("case_num", "n_component"),
                            gp_models[goal - 1]
                            .predict(
                                dataset=era5_df_for_goal,
                            )
                            .values,
                        )
                    },
                    coords={"case_num": era5_df_for_goal.index},
                )
            )
            prediction_ds_in_buoys = xr.Dataset(
                {
                    "hs": (
                        ("time", "stations"),
                        prediction_ds_in_points.point_hm0.values,
                    ),
                    "tp": (
                        ("time", "stations"),
                        prediction_ds_in_points.point_tp.values,
                    ),
                    "dp": (
                        ("time", "stations"),
                        get_degrees_from_uv(
                            prediction_ds_in_points.point_wavdir_u.values,
                            prediction_ds_in_points.point_wavdir_v.values,
                        ),
                    ),
                    "spr": (
                        ("time", "stations"),
                        era5_df_for_goal.ds.repeat(177).values.reshape(744, 177),
                    ),
                },
                coords={
                    "time": era5_df_for_goal.index,
                },
            ).expand_dims({"goal": [goal], "partition": [partition]})
            prediction_ds_in_buoys["hs"] = (
                prediction_ds_in_buoys.hs * era5_ds_for_goal.hs
            )
            prediction_ds_in_buoys = prediction_ds_in_buoys.drop(
                ["longitude", "latitude"]
            )

            # Save the prediction in buoys
            reconstruction_for_each_goal.append(prediction_ds_in_buoys)

    #     # Concatenate the partitions and save as variable
    #     partition_hm0_in_buoys = np.sqrt(
    #         (xr.concat(reconstruction_for_each_goal, dim="goal").hm0 ** 2).sum(
    #             dim="goal"
    #         )
    #     )
    #     reconstructed_partitions.append(partition_hm0_in_buoys)

    # metamodel_hm0_in_buoys = xr.Dataset(
    #     {
    #         f"{partition}_hm0": partition_hm0
    #         for partition, partition_hm0 in zip(
    #             ["bulk", "sea", "swell1", "swell2", "swell3"],
    #             reconstructed_partitions,
    #         )
    #     }
    # )
    # metamodel_hm0_in_buoys["sum_hm0"] = np.sqrt(
    #     metamodel_hm0_in_buoys.sea_hm0**2
    #     + metamodel_hm0_in_buoys.swell1_hm0**2
    #     + metamodel_hm0_in_buoys.swell2_hm0**2
    #     + metamodel_hm0_in_buoys.swell3_hm0**2
    # )
    # reconstructed_months.append(metamodel_hm0_in_buoys)
    # metamodel_hm0_in_buoys.to_netcdf(f"outputs/1500_reconstructed_month_{month_str}.nc")

INFO:ExactGPInterpolation:Preprocessing dataset for prediction
INFO:ExactGPInterpolation:Checking for NaNs in subset data
INFO:ExactGPInterpolation:Preprocessing subset data
INFO:ExactGPInterpolation:Normalizing subset data
INFO:ExactGPInterpolation:Proposed min custom scaler for wd_u is bigger than datapoint
INFO:ExactGPInterpolation:Proposed max custom scaler for wd_u is lower than datapoint
INFO:ExactGPInterpolation:Proposed min custom scaler for wd_v is bigger than datapoint
INFO:ExactGPInterpolation:Subset data preprocessed successfully
INFO:ExactGPInterpolation:Predicting target variable: PC1
INFO:ExactGPInterpolation:Predicting target variable: PC2
INFO:ExactGPInterpolation:Predicting target variable: PC3
INFO:ExactGPInterpolation:Predicting target variable: PC4
INFO:ExactGPInterpolation:Predicting target variable: PC5
INFO:ExactGPInterpolation:Predicting target variable: PC6
INFO:ExactGPInterpolation:Predicting target variable: PC7
INFO:ExactGPInterpolation:Predicting target va

In [13]:
datasesito = xr.merge(reconstruction_for_each_goal).isel(stations=slice(-9, None))
datasesito

<xarray.Dataset> Size: 11MB
Dimensions:    (goal: 10, partition: 5, time: 744, stations: 9)
Coordinates:
  * goal       (goal) int64 80B 1 2 3 4 5 6 7 8 9 10
  * partition  (partition) object 40B 'bulk' 'sea' 'swell1' 'swell2' 'swell3'
  * time       (time) datetime64[ns] 6kB 2011-12-01 ... 2011-12-31T23:00:00
Dimensions without coordinates: stations
Data variables:
    hs         (goal, partition, time, stations) float64 3MB -0.1895 ... -2.8...
    tp         (goal, partition, time, stations) float64 3MB 5.949 ... 3.46
    dp         (goal, partition, time, stations) float64 3MB 263.0 ... 19.5
    spr        (goal, partition, time, stations) float64 3MB 31.42 31.42 ... 5.0

In [14]:
partitions = (
    datasesito.isel(partition=[1, 2, 3, 4])
    .rename({"stations": "site"})
    .stack(part=("goal", "partition"))
)
partitions

<xarray.Dataset> Size: 9MB
Dimensions:    (time: 744, site: 9, part: 40)
Coordinates:
  * time       (time) datetime64[ns] 6kB 2011-12-01 ... 2011-12-31T23:00:00
  * part       (part) object 320B MultiIndex
  * goal       (part) int64 320B 1 1 1 1 2 2 2 2 3 3 ... 8 8 9 9 9 9 10 10 10 10
  * partition  (part) object 320B 'sea' 'swell1' 'swell2' ... 'swell2' 'swell3'
Dimensions without coordinates: site
Data variables:
    hs         (time, site, part) float64 2MB -0.2485 0.01456 ... -2.852e-06
    tp         (time, site, part) float64 2MB 5.358 9.289 17.51 ... 15.95 3.46
    dp         (time, site, part) float64 2MB 226.1 282.0 332.3 ... 239.6 19.5
    spr        (time, site, part) float64 2MB 22.6 5.0 5.0 5.0 ... 5.0 5.0 5.0

In [15]:
from wavespectra.construct import construct_partition

dir = np.arange(0, 360, 5)
freq = np.logspace(np.log10(0.035), np.log10(0.5), 29)
full_spec = construct_partition(
    freq_name="jonswap",
    freq_kwargs={
        "freq": freq,
        "fp": 1 / partitions.tp,
        "hs": partitions.hs,
    },
    dir_name="cartwright",
    dir_kwargs={
        "dir": dir,
        "dm": partitions.dp,
        "dspr": partitions.spr,
    },
)
full_spec

<xarray.DataArray (time: 744, site: 9, part: 40, freq: 29, dir: 72)> Size: 4GB
array([[[[[0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
          [4.09736633e-310, 3.07673280e-311, 1.59741188e-312, ...,
           1.99176207e-307, 3.14966371e-308, 4.05802765e-309],
          [1.98604502e-215, 1.49133111e-216, 7.74285638e-218, ...,
           9.65432138e-213, 1.52668164e-213, 1.96697707e-214],
          ...,
          [4.58580841e-015, 3.44350639e-016, 1.78783741e-017, ...,
           2.22919761e-012, 3.52513131e-013, 4.54178022e-014],
          [2.89937926e-015, 2.17715834e-016, 1.13036094e-017, ...,
           1.40941111e-012, 2.22876572e-013, 2.87154241e-014],
          [1.82365738e-015, 1.36939341e-016, 7.10976689e-018, ...,
           8.86494222e-013, 1.40185353e-013, 1.80614850e-014]],

         [[1.28589484e-108, 5.22183128e-117, 3.58697259e-126, ...,
           1.45875007e-087, 6.14884467e-094, 6.07998535e-101],
          [1.83074544e-093, 7.43439001e-102, 5.10682015e-111, ...,
           2.07684170e-072, 8.75419120e-079, 8.65615528e-086],
          [3.64234595e-083, 1.47910353e-091, 1.01602360e-100, ...,
           4.13196495e-062, 1.74168359e-068, 1.72217893e-075],
...
          [9.12093417e-167, 5.29340393e-185, 2.64708500e-205, ...,
           1.06443161e-121, 2.50450513e-135, 2.74802023e-150],
          [5.67410702e-167, 3.29301142e-185, 1.64674399e-205, ...,
           6.62179858e-122, 1.55804546e-135, 1.70953552e-150],
          [3.52961274e-167, 2.04843776e-185, 1.02436711e-205, ...,
           4.11913004e-122, 9.69191648e-136, 1.06342695e-150]],

         [[0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
          [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
          [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
          ...,
          [2.76838151e-017, 8.40917972e-016, 9.27871167e-015, ...,
           1.94393970e-024, 1.35772365e-021, 3.26680305e-019],
          [1.89200022e-017, 5.74710162e-016, 6.34136749e-015, ...,
           1.32855039e-024, 9.27911646e-022, 2.23263740e-019],
          [1.25511721e-017, 3.81251868e-016, 4.20674344e-015, ...,
           8.81335241e-025, 6.15559062e-022, 1.48108949e-019]]]]])
Coordinates:
  * time       (time) datetime64[ns] 6kB 2011-12-01 ... 2011-12-31T23:00:00
  * part       (part) object 320B MultiIndex
  * freq       (freq) float64 232B 0.035 0.03849 0.04232 ... 0.4135 0.4547 0.5
  * dir        (dir) int64 576B 0 5 10 15 20 25 30 ... 330 335 340 345 350 355
  * goal       (part) int64 320B 1 1 1 1 2 2 2 2 3 3 ... 8 8 9 9 9 9 10 10 10 10
  * partition  (part) object 320B 'sea' 'swell1' 'swell2' ... 'swell2' 'swell3'
Dimensions without coordinates: site
Attributes: (12/30)
    GRIB_NV:                                  0
    GRIB_Nx:                                  720
    GRIB_Ny:                                  361
    GRIB_cfName:                              unknown
    GRIB_cfVarName:                           swh
    GRIB_dataType:                            an
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_typeOfLevel:                         meanSea
    GRIB_units:                               m
    GRIB_uvRelativeToGrid:                    0
    long_name:                                Significant height of combined ...
    units:                                    m

In [20]:
full_spec.sum(dim="part").to_netcdf("outputs/201112_500_spectra.nc")

In [ ]:
month_ds = [
    xr.open_dataset(os.path.join("outputs/reconstructed_months/", file))
    for file in sorted(os.listdir("outputs/reconstructed_months/"))
]

metamodel_all_months = xr.concat(month_ds, dim="time")
metamodel_all_months

In [ ]:
metamodel_all_months.to_netcdf(
    "outputs/Metamodel_Reconstruction_in_Points_2000_2018.nc"
)